In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
import matplotlib.pyplot as plt
import datetime
import numpy as np
import time

In [3]:
import pyearthtools.data as petdata
from pyearthtools.data.exceptions import DataNotFoundError
import pyearthtools.pipeline as petpipe
import site_archive_jasmin # Required to not get missing something error

ROOT_DIRECTORIES: {'ERA5lowres': '/gws/ssde/j25a/mmh_storage/theme3/weatherbench/5.625deg/', 'MOGLOBAL': '/gws/ssde/j25a/mmh_storage/theme3/mo_pet_site_archive/mo_global/', 'MOUKV': '/gws/ssde/j25a/mmh_storage/theme3/mo_pet_site_archive/mo_ukv/', 'Himawari': '/gws/ssde/j25a/mmh_storage/theme3/rv74_himawari', 'HimawariChannels': '/gws/ssde/j25a/mmh_storage/theme3/ra22_himawari', 'Rainfields3': '/gws/ssde/j25a/mmh_storage/theme3/rq0Radar'}


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

In [5]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [6]:
selected_date = datetime.datetime(2021,6,9,2,0)
himawari = petdata.archive.Himawari('surface_global_irradiance')
rf3proj = petdata.transforms.projection.Rainfields3ProjAus()
radar_projector = petdata.transforms.projection.XYtoLonLatRectilinear(rf3proj)
satpipe = petpipe.Pipeline(
    himawari
)


In [7]:
fullsat = petpipe.Pipeline(
    satpipe,
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),  # 
    # Align the data variable's coordinate order to the dataset coordinate order so all arrays are the same shape
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),  
    petdata.transform.region.Bounding(-35, -25, 138, 150),  # cut down on region for example
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    iterator=petpipe.iterators.DateRange('20200101T00', '20210101T00', interval='1 day'),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError
)

In [8]:
# Reminder, the image size is latitude: 1726, longitude: 2214

class AutoEncoder(nn.Module):
    def __init__(self, 
                 input_height = 501,
                 input_width = 601,
                 kernel_size = 4,
                 stride=2,
                 input_channel_count = 2,
                 output_channel_count = 2,
                 latent_dim=300):
        super(AutoEncoder, self).__init__()

        self.input_width = input_width
        self.input_height = input_height
        self.input_channel_count = input_channel_count
        self.output_channel_count = output_channel_count

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=self.input_channel_count, out_channels=16, kernel_size=kernel_size, stride = stride, padding = 1),
            nn.ReLU(),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride =2, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=7),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=7),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=16, out_channels=self.output_channel_count, kernel_size=kernel_size, stride=stride, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):

        # Get latent representation
        latent = self.encoder(x)

        # Reconstruct input
        reconstructed = self.decoder(latent)

        return reconstructed

In [9]:
model = AutoEncoder(input_channel_count=1, output_channel_count=1).to(device)

In [10]:
# Loss function and optimizer
criterion = nn.L1Loss()
# criterion = nn.KLDivLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [11]:
import asyncio

async def load_sample(pipeline, sample_id):
    def sample():
        return pipeline[sample_id] # Unique per thread
        # return np.array([1, 2, 3, 4]) # Doesn't crash kernel when just returning a simple np array
    
    try:
        task_start_time = time.time()

        blocking_coroutine = asyncio.to_thread(sample)
        sample_task = asyncio.create_task(blocking_coroutine)
        
        sample = await sample_task
        
        print(f"Sample load time {sample_id} {time.time() - task_start_time}s")

        return sample
    except DataNotFoundError:
        return None
    except Exception as e:
        raise e

async def get_next_batch(pipeline, sample_id_iterator, sample_ix, max_samples, batch_size):
    tasks = set()
    batch = []
    while True:
        ########################### Adding sample task ###########################
        try:
            sample_id = next(sample_id_iterator)
        except StopIteration: # If any remaining in batch, don't process
            break

        tasks.add(asyncio.create_task(load_sample(pipeline, sample_id)))

        ########################### Checking if full samples ###########################
        if len(batch) + len(tasks) == batch_size:
            finished, unfinished = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
            print(f"{len(finished)} finished tasks, {len(unfinished)} unfinished tasks")

            for idx, task in enumerate(finished):
                sample = task.result()

                if sample is None:
                    continue

                if torch.isnan(torch.tensor(sample)).any():
                    continue

                batch.append(sample[0])
                sample_ix += 1

            print(f"Tasks length {len(tasks)}")
            tasks = unfinished
            print(f"Tasks length again{len(tasks)}")

            # The batch has been populated with the requird number of samples
            if len(batch) == batch_size:
                break

        # if sample_ix > max_samples:
        #     break
        
    return np.stack(batch)

In [12]:
async def train(debug=True, num_epochs=1, max_samples=10, print_per=20, batch_size=16):
    """
    Main Training loop Function
    """
    print("a")
    
    sample_ix = 0
    for epoch in range(num_epochs):
        total_loss = 0
        epoch_samples = 0
        sample_date_iterator = iter(fullsat.iteration_order)
        print("b")

        next_batch_task = asyncio.create_task(get_next_batch(fullsat, sample_date_iterator, sample_ix, max_samples, batch_size))
        print("c")
        start_time = time.time()
        while True:         
            task_start_time = time.time()
            batch = await next_batch_task
            batch_load_time = time.time() - task_start_time
            print("d")

            next_batch_task = asyncio.create_task(get_next_batch(fullsat, sample_date_iterator, sample_ix, max_samples, batch_size))
            
            sample_ix += batch.shape[0]
            epoch_samples += 1
            print(sample_ix)

            x = torch.from_numpy(batch).float().to(device)

            print("e")

            optimizer.zero_grad()

            # Forward pass
            y = model.forward(x)
            loss = criterion(y, x)
    
            # Backward pass and optimize        
            loss.backward()
            optimizer.step()
    
            total_loss += loss.item()

            batch = []

            if epoch_samples % print_per == 0:
                sample_train_run_time = time.time() - start_time
                print(f"[Epoch {epoch+1}] Sample {epoch_samples}, Batch Loss: {loss.item():.4f}, Load wait time: {batch_load_time:.2f}, Actual Train time: {sample_train_run_time - batch_load_time:.2f}")
                start_time = time.time()
    
        # Print epoch statistics
        avg_loss = total_loss / epoch_samples
        epoch_samples = 0  # Reset for next epoch
        print(f'Epoch [{epoch+1}/{epoch_samples}], Average Loss: {avg_loss:.4f}')

In [13]:
# Note %%time will not work with this use of await
await train(debug=False, num_epochs=1, max_samples=5000, print_per = 1, batch_size=1)


a
b
c


/home/users/train097/PyEarthTools/packages/data/src/pyearthtools/data/operations/index_routines.py:326: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  full_ds = xr.open_mfdataset(


Sample load time 2020-01-01T00 1.0447382926940918s
1 finished tasks, 0 unfinished tasks
Tasks length 1
Tasks length again0
d
1
e
[Epoch 1] Sample 1, Batch Loss: 0.2806, Load wait time: 1.05, Actual Train time: 0.35


/home/users/train097/PyEarthTools/packages/data/src/pyearthtools/data/operations/index_routines.py:326: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  full_ds = xr.open_mfdataset(


Sample load time 2020-01-02T00 0.5485248565673828s
1 finished tasks, 0 unfinished tasks
Tasks length 1
Tasks length again0
d
2
e
[Epoch 1] Sample 2, Batch Loss: 0.2774, Load wait time: 0.55, Actual Train time: 0.01


...

/home/users/train097/PyEarthTools/packages/data/src/pyearthtools/data/operations/index_routines.py:326: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  full_ds = xr.open_mfdataset(


Sample load time 2020-03-02T00 0.6093425750732422s
1 finished tasks, 0 unfinished tasks
Tasks length 1
Tasks length again0
d
60
e
[Epoch 1] Sample 60, Batch Loss: 0.0752, Load wait time: 0.61, Actual Train time: 0.01


/home/users/train097/PyEarthTools/packages/data/src/pyearthtools/data/operations/index_routines.py:326: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  full_ds = xr.open_mfdataset(


Sample load time 2020-03-03T00 0.7089254856109619s


CancelledError: 

In [ ]:
fullsat_validate = petpipe.Pipeline(
    satpipe,
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),  # 
    # Align the data variable's coordinate order to the dataset coordinate order so all arrays are the same shape
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),  
    petdata.transform.region.Bounding(-35, -25, 138, 150),  # cut down on region for example
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    iterator=petpipe.iterators.DateRange('20210101T00', '20220101T00', interval='10 minutes'),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError
)

In [ ]:
training_iterator = iter(fullsat) 
sample_numpy = next(training_iterator)

validate_iterator = iter(fullsat_validate)

def next_sample_gpu(pet_iterator):
    return torch.from_numpy(next(pet_iterator)).float().to(device)

sample_tensor_gpu = torch.from_numpy(sample_numpy).float().to(device)
sample_prediction_gpu = model.forward(sample_tensor_gpu)
sample_prediction_gpu = model.forward(next_sample_gpu(training_iterator))

def prediction_from_iterator(pet_iterator, model):
    sample_numpy = next(pet_iterator)
    sample_gpu = torch.from_numpy(sample_numpy).float().to(device)
    prediction_gpu = model.forward(sample_gpu)
    prediction_cpu = sample_prediction_gpu.to('cpu').detach().numpy()
    return prediction_cpu

prediction_from_iterator(validate_iterator, model)

In [ ]:
fig1 = plt.figure('sidebyside_satellite', figsize=(24,8))
for ix1 in range(3):
    sample_numpy = next(validate_iterator)
    sample_gpu = torch.from_numpy(sample_numpy).float().to(device)
    prediction_gpu = model.forward(sample_gpu)
    prediction_cpu = prediction_gpu.to('cpu').detach().numpy()
    ax1 = fig1.add_subplot(3,3,(ix1*3)+1)
    ax1.imshow(sample_numpy[0,0])
    ax1 = fig1.add_subplot(3,3,(ix1*3)+2)
    ax1.imshow(prediction_cpu[0,0])
    ax1 = fig1.add_subplot(3,3,(ix1*3)+3)
    ax1.imshow(prediction_cpu[0,0] - sample_numpy[0,0])  

In [ ]:
import torchmetrics.image
ssi_metric = torchmetrics.image.StructuralSimilarityIndexMeasure()
rmse_sw_metric = torchmetrics.image.RootMeanSquaredErrorUsingSlidingWindow()

ssi_values = []
rmse_values = []
for ix1 in range(10):
    sample_numpy = next(validate_iterator)
    sample_cpu = torch.from_numpy(sample_numpy).float()
    sample_gpu = sample_cpu.to(device)
    prediction_gpu = model.forward(sample_gpu)
    prediction_cpu_tensor = prediction_gpu.to('cpu').detach()
    
    ssi_values += [float(ssi_metric(sample_cpu,  prediction_cpu_tensor)) ]
    rmse_values += [float(rmse_sw_metric(sample_cpu,  prediction_cpu_tensor))]

def mean(list_):
    print(type(list_))
    return sum(list_)/len(list_)

print(f"Mean SSE {mean(ssi_values)}")
print(f"Mean RMSE {mean(rmse_values)}")